# M3: RGB-only self-supervised PoseNet + DepthNet

Pin a reviewed, pushed M3 commit. The standalone M3 branch supports fresh training
and pose evaluation using the existing shared infrastructure.
All model code comes from Git; do not edit architecture in notebook cells.
Nothing trains or evaluates by default: opt in independently with RUN_PILOT,
RUN_FULL and EVALUATE. Inspect pilot coverage, updates and peak memory before
starting the full run. Test evaluation is opt-in after the configuration is frozen.
Checkpoint resume and depth evaluation require their separately reviewed shared
integrations. Depth is unnecessary for training and pose-only evaluation.
Keep this notebook output-free and never commit checkpoints.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing"
print("torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))
REPO = "VaibhavBansal05/Deep_Monocular_VO"
REVISION = "REPLACE_WITH_REVIEWED_COMMIT"  # full 40-character commit SHA
CONFIG = "configs/m3.yaml"  # matched ablations: configs/m3_* yaml
RUN_PILOT = False  # explicitly enable a short GPU pilot
PILOT_STEPS = 200
RUN_FULL = False
EVALUATE = False  # enable only after freezing config and selecting validation checkpoint
EVAL_CKPT = None  # optional checkpoint path for evaluation-only; defaults to OUT/best.pt
EVAL_DEPTH = False  # requires the separately reviewed depth-evaluation integration
DEPTH_ROOT = None  # optional separate dataset root containing Scene02/Scene20 depth trees
NUM_WORKERS = 2
assert re.fullmatch(r"[0-9a-f]{40}", REVISION), "Set REVISION to your reviewed/pushed commit"


In [ ]:
from google.colab import drive, userdata
import getpass, shutil
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/DL_PROJECT")  # team folder
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = getpass.getpass("GitHub token: ")
WORK = Path("/content")
INPUT = "/content/m3-input"
Path(INPUT).mkdir(exist_ok=True)
# Copy the archive off the Drive FUSE mount before extracting.
archive = Path(INPUT) / "vkitti2_subset.tar.gz"
if not archive.exists(): shutil.copy(DRIVE / archive.name, archive)
CHECKPOINT_ROOT = DRIVE / "checkpoints"
RESULT_ROOT = DRIVE / "results"


In [ ]:
import tempfile
repo_path = WORK / "Deep_Monocular_VO"
# Askpass keeps the token out of the URL, saved cells and command error messages.
with tempfile.NamedTemporaryFile(mode="w", suffix=".sh", delete=False) as f:
    f.write('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$M3_GIT_TOKEN" ;; esac\n')
    askpass = f.name
os.chmod(askpass, 0o700)
environment = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0", M3_GIT_TOKEN=TOKEN)
try:
    if not repo_path.exists():
        subprocess.run(["git", "clone", "--quiet", f"https://github.com/{REPO}.git", str(repo_path)], env=environment, check=True)
    else:
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_path, text=True).strip(), "Checkout contains changes; use a fresh runtime directory"
        subprocess.run(["git", "fetch", "origin"], cwd=repo_path, env=environment, check=True)
    subprocess.run(["git", "checkout", "--detach", REVISION], cwd=repo_path, check=True)
finally:
    os.unlink(askpass)
    environment.pop("M3_GIT_TOKEN", None)
    del TOKEN
SRC = repo_path / "src"
os.chdir(SRC)
os.environ["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
# Preserve the platform's compatible torch/torchvision wheels.
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(WORK / "m3-venv")], check=True)
PYTHON = str(WORK / "m3-venv/bin/python")
subprocess.run([PYTHON, "-m", "pip", "install", "-e", ".[dev]", "--no-deps"], check=True)
def run(*args):
    subprocess.run([PYTHON, *args], check=True)
run("-m", "pytest", "-q", "-p", "no:cacheprovider")
# Feature detection keeps the standalone M3 branch usable without the resume PR.
HAS_CHECKPOINT_RESUME = subprocess.check_output(
    [PYTHON, "-c", "from mvo.common.train import engine; print(int(callable(getattr(engine, 'restore_rng', None))))"],
    text=True).strip() == "1"
print("Checkpoint resume integration:", "available" if HAS_CHECKPOINT_RESUME else "unavailable; fresh training supported")


In [ ]:
import glob, hashlib, shutil
roots = glob.glob(INPUT + "/**/vkitti2/Scene20/clone/extrinsic.txt", recursive=True)
tars = glob.glob(INPUT + "/**/vkitti2_subset.tar.gz", recursive=True)
if roots:
    DATA_ROOT = Path(roots[0]).parents[2]
elif tars:
    archive = Path(tars[0])
    assert archive.stat().st_size == 1514968986, "RGB archive is truncated"
    digest = hashlib.md5()
    with archive.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    assert digest.hexdigest() == "aca1947cc25aaa968c7d6c07d3c3ff01", "RGB checksum mismatch"
    data = WORK / "data"
    data.mkdir(exist_ok=True)
    subprocess.run(["tar", "xzf", str(archive), "-C", str(data)], check=True)
    DATA_ROOT = data / "vkitti2"
else:
    raise RuntimeError("Attach vkitti2_subset.tar.gz or the extracted vkitti2 tree")
assert len(list(DATA_ROOT.rglob("rgb_*.jpg"))) == 8642, "Incomplete RGB subset"
if DEPTH_ROOT:
    overlay = WORK / "depth-overlay/vkitti2"
    for scene in DATA_ROOT.glob("Scene*"):
        for variation in scene.iterdir():
            if not (variation / "frames/rgb/Camera_0").is_dir():
                continue
            dest = overlay / scene.name / variation.name
            (dest / "frames").mkdir(parents=True, exist_ok=True)
            for text in variation.glob("*.txt"):
                link = dest / text.name
                if not link.exists(): link.symlink_to(text)
            link = dest / "frames/rgb"
            if not link.exists(): link.symlink_to(variation / "frames/rgb", target_is_directory=True)
            depth = Path(DEPTH_ROOT) / scene.name / variation.name / "frames/depth"
            link = dest / "frames/depth"
            if depth.exists() and not link.exists(): link.symlink_to(depth, target_is_directory=True)
    DATA_ROOT = overlay
if EVALUATE and EVAL_DEPTH:
    for sid, n in [("Scene02/clone", 233), ("Scene20/clone", 837)]:
        d = DATA_ROOT / sid / "frames/depth/Camera_0"
        assert all((d / f"depth_{i:05d}.png").is_file() for i in range(n)), f"Missing depth for {sid}"
print("data root", DATA_ROOT)


## Optional pilot: set RUN_PILOT=True


In [ ]:
RUN_NAME = Path(CONFIG).stem
OUT = CHECKPOINT_ROOT / RUN_NAME
PILOT = WORK / "pilot" / RUN_NAME
if RUN_PILOT:
    if not isinstance(PILOT_STEPS, int) or PILOT_STEPS <= 0:
        raise ValueError("PILOT_STEPS must be a positive integer")
    run("-m", "mvo.models.m3_selfsup_posedepth.diagnostics", "smoke", "--config", CONFIG,
        "--data-root", str(DATA_ROOT), "--out", str(PILOT), "--steps", str(PILOT_STEPS), "--device", "cuda")
    run("-m", "mvo.models.m3_selfsup_posedepth.diagnostics", "visualize",
        "--ckpt", str(PILOT / "best.pt"), "--data-root", str(DATA_ROOT), "--out", str(PILOT))
    print((PILOT / "smoke.json").read_text())
else:
    print("Pilot skipped. Set RUN_PILOT=True to request it.")


## Full run and validation-selected evaluation


In [ ]:
if RUN_FULL:
    args = ["scripts/train.py", "--model", "m3", "--config", CONFIG,
            "--data-root", str(DATA_ROOT), "--out", str(OUT), "--num-workers", str(NUM_WORKERS)]
    if (OUT / "last.pt").exists():
        if not HAS_CHECKPOINT_RESUME:
            raise RuntimeError("Existing last.pt found without the reviewed resume integration; use a fresh output directory for standalone training.")
        args += ["--resume", str(OUT / "last.pt")]
    run(*args)
else:
    print("Full training skipped. Set RUN_FULL=True to request it.")


In [ ]:
if EVALUATE:
    checkpoint = Path(EVAL_CKPT) if EVAL_CKPT else OUT / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Evaluation checkpoint missing: {checkpoint}; set EVAL_CKPT or restore OUT/best.pt")
    if EVAL_DEPTH:
        help_text = subprocess.check_output([PYTHON, "scripts/eval.py", "--help"], text=True)
        if "--eval-depth" not in help_text:
            raise RuntimeError("Depth evaluation requires the separately reviewed depth-evaluation integration; use EVAL_DEPTH=False for pose-only evaluation.")
    RESULTS = RESULT_ROOT / RUN_NAME
    args = ["scripts/eval.py", "--model", "m3", "--ckpt", str(checkpoint),
            "--data-root", str(DATA_ROOT), "--out", str(RESULTS)]
    if EVAL_DEPTH: args += ["--eval-depth"]
    run(*args)
    run("-m", "mvo.models.m3_selfsup_posedepth.diagnostics", "visualize",
        "--ckpt", str(checkpoint), "--data-root", str(DATA_ROOT), "--split", "test",
        "--out", str(RESULTS), *(["--with-depth"] if EVAL_DEPTH else []))
    history = checkpoint.parent / "history.csv"
    if history.is_file(): shutil.copy(history, RESULTS / "history.csv")
    print((RESULTS / "metrics.json").read_text())
    # Weather/viewpoint tests are separate output directories; depth is not required.
    run("scripts/eval.py", "--model", "m3", "--ckpt", str(checkpoint),
        "--data-root", str(DATA_ROOT), "--split", "robustness", "--out", str(RESULTS / "robustness"))
